# Experiment Analysis

Statistical analysis of A/B experiment results: t-tests, chi-squared, confidence intervals,
effect sizes, power analysis, and segment breakdowns.

**Outputs:** `experiment-results-{date}.json`, `experiment-summary-{date}.md`, 4 PNG charts

In [ ]:
# Papermill parameters
experiment_name = "unnamed-experiment"
control_data_path = ""  # Path to control group CSV or BQ query
treatment_data_path = ""  # Path to treatment group CSV or BQ query
primary_metric = "conversion_rate"  # Column name for primary metric
guardrail_metrics = ""  # Comma-separated column names
significance_level = 0.05
output_dir = None
report_date = "2026-04-04"

In [ ]:
import sys
from pathlib import Path

# Find workspace root (walk up to .git), then add data/notebooks/ to sys.path
# so that utils.pm_helpers is importable regardless of cwd
# (papermill runs notebooks from outputs/, not templates/)
_dir = Path.cwd()
while _dir != _dir.parent:
    if (_dir / ".git").exists():
        break
    _dir = _dir.parent
WORKSPACE_ROOT = _dir
NOTEBOOKS_DIR = WORKSPACE_ROOT / "data" / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
from utils.pm_helpers import (
    set_chart_style,
    save_chart,
    save_json,
    BRAND_COLOURS,
    PALETTE,
)

set_chart_style()

# Parse guardrail metrics
guardrail_list = [m.strip() for m in guardrail_metrics.split(",") if m.strip()] if guardrail_metrics else []

print(f"Experiment: {experiment_name}")
print(f"Primary metric: {primary_metric}")
print(f"Guardrails: {guardrail_list or 'none'}")
print(f"Significance level: {significance_level}")

In [ ]:
# --- Load control and treatment data ---

def load_experiment_data(path_str):
    """Load data from CSV path."""
    if not path_str:
        return None
    p = Path(path_str)
    if p.suffix == ".csv" and p.exists():
        return pd.read_csv(p)
    # Try relative to workspace
    rel = WORKSPACE_ROOT / path_str
    if rel.exists():
        return pd.read_csv(rel)
    print(f"WARNING: Could not load data from {path_str}")
    return None

control = load_experiment_data(control_data_path)
treatment = load_experiment_data(treatment_data_path)

if control is not None and treatment is not None:
    print(f"Control: {len(control)} observations, Treatment: {len(treatment)} observations")
    print(f"Control columns: {list(control.columns)}")
else:
    print("Data not loaded — provide valid control_data_path and treatment_data_path.")
    print("This notebook can still serve as a template for manual analysis.")

In [ ]:
# --- Statistical tests ---

results = {"experiment": experiment_name, "report_date": report_date, "tests": {}}

def run_test(control_vals, treatment_vals, metric_name, alpha=0.05):
    """Run appropriate statistical test and return results dict."""
    c = control_vals.dropna()
    t = treatment_vals.dropna()

    result = {
        "metric": metric_name,
        "control_n": len(c),
        "treatment_n": len(t),
        "control_mean": float(c.mean()),
        "treatment_mean": float(t.mean()),
        "absolute_delta": float(t.mean() - c.mean()),
        "relative_delta_pct": float((t.mean() - c.mean()) / c.mean() * 100) if c.mean() != 0 else None,
    }

    # Determine test type
    unique_vals = set(c.unique()) | set(t.unique())
    is_binary = unique_vals <= {0, 1, 0.0, 1.0}

    if is_binary:
        # Chi-squared test for proportions
        c_success, c_total = int(c.sum()), len(c)
        t_success, t_total = int(t.sum()), len(t)
        contingency = np.array([[c_success, c_total - c_success], [t_success, t_total - t_success]])
        chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
        result["test"] = "chi-squared"
        result["chi2"] = round(float(chi2), 4)
        result["p_value"] = round(float(p_value), 6)
        # Effect size (Cohen's h for proportions)
        p1 = c_success / c_total
        p2 = t_success / t_total
        cohens_h = 2 * np.arcsin(np.sqrt(p2)) - 2 * np.arcsin(np.sqrt(p1))
        result["effect_size_cohens_h"] = round(float(cohens_h), 4)
    else:
        # Welch's t-test (unequal variances)
        t_stat, p_value = stats.ttest_ind(c, t, equal_var=False)
        result["test"] = "welch_t"
        result["t_statistic"] = round(float(t_stat), 4)
        result["p_value"] = round(float(p_value), 6)
        # Cohen's d
        pooled_std = np.sqrt((c.std()**2 + t.std()**2) / 2)
        cohens_d = (t.mean() - c.mean()) / pooled_std if pooled_std > 0 else 0
        result["effect_size_cohens_d"] = round(float(cohens_d), 4)

    # 95% confidence interval for the difference in means
    se = np.sqrt(c.var() / len(c) + t.var() / len(t))
    z_crit = stats.norm.ppf(1 - alpha / 2)
    diff = t.mean() - c.mean()
    result["ci_lower"] = round(float(diff - z_crit * se), 6)
    result["ci_upper"] = round(float(diff + z_crit * se), 6)
    result["significant"] = p_value < alpha

    # Power analysis (post-hoc)
    if pooled_std if not is_binary else se > 0:
        from scipy.stats import norm
        effect = abs(diff) / (pooled_std if not is_binary else se)
        z_alpha = norm.ppf(1 - alpha / 2)
        power = 1 - norm.cdf(z_alpha - effect * np.sqrt(min(len(c), len(t)) / 2))
        result["power"] = round(float(power), 4)

    # MDE (minimum detectable effect at 80% power)
    z_beta = norm.ppf(0.8) if 'norm' in dir() else stats.norm.ppf(0.8)
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    n_min = min(len(c), len(t))
    if n_min > 0 and se > 0:
        mde = (z_alpha + z_beta) * se * np.sqrt(2)
        result["mde_absolute"] = round(float(mde), 6)
        if c.mean() != 0:
            result["mde_relative_pct"] = round(float(mde / abs(c.mean()) * 100), 2)

    return result

if control is not None and treatment is not None:
    # Primary metric
    if primary_metric in control.columns and primary_metric in treatment.columns:
        primary_result = run_test(control[primary_metric], treatment[primary_metric], primary_metric, significance_level)
        results["tests"]["primary"] = primary_result
        print(f"\n=== Primary Metric: {primary_metric} ===")
        print(f"Control mean: {primary_result['control_mean']:.4f} (n={primary_result['control_n']})")
        print(f"Treatment mean: {primary_result['treatment_mean']:.4f} (n={primary_result['treatment_n']})")
        print(f"Delta: {primary_result['absolute_delta']:.4f} ({primary_result.get('relative_delta_pct', 'N/A')}%)")
        print(f"p-value: {primary_result['p_value']:.6f} ({'SIGNIFICANT' if primary_result['significant'] else 'not significant'})")
        print(f"95% CI: [{primary_result['ci_lower']:.4f}, {primary_result['ci_upper']:.4f}]")
    else:
        print(f"WARNING: Primary metric '{primary_metric}' not found in data.")

    # Guardrail metrics
    results["tests"]["guardrails"] = []
    for gm in guardrail_list:
        if gm in control.columns and gm in treatment.columns:
            gr = run_test(control[gm], treatment[gm], gm, significance_level)
            results["tests"]["guardrails"].append(gr)
            status = "BREACHED" if gr["significant"] and gr["absolute_delta"] < 0 else "SAFE"
            print(f"\nGuardrail: {gm} — {status}")
            print(f"  Delta: {gr['absolute_delta']:.4f}, p={gr['p_value']:.4f}")
else:
    print("No data — skipping statistical tests.")

In [ ]:
# --- Segment breakdown ---

segment_results = []
segment_cols = ["device", "value_band", "user_type", "customer_segment", "segment", "platform"]

if control is not None and treatment is not None and primary_metric in control.columns:
    for seg_col in segment_cols:
        if seg_col in control.columns and seg_col in treatment.columns:
            print(f"\n=== Segment: {seg_col} ===")
            all_segments = set(control[seg_col].dropna().unique()) | set(treatment[seg_col].dropna().unique())
            for seg_val in sorted(all_segments):
                c_seg = control[control[seg_col] == seg_val][primary_metric]
                t_seg = treatment[treatment[seg_col] == seg_val][primary_metric]
                if len(c_seg) >= 10 and len(t_seg) >= 10:
                    sr = run_test(c_seg, t_seg, f"{primary_metric}__{seg_col}={seg_val}", significance_level)
                    sr["segment_column"] = seg_col
                    sr["segment_value"] = str(seg_val)
                    segment_results.append(sr)
                    sig = "*" if sr["significant"] else ""
                    print(f"  {seg_val}: delta={sr['absolute_delta']:.4f}, p={sr['p_value']:.4f}{sig} (n={sr['control_n']}+{sr['treatment_n']})")
                else:
                    print(f"  {seg_val}: insufficient sample (n={len(c_seg)}+{len(t_seg)})")

    results["tests"]["segments"] = segment_results
    if segment_results:
        print(f"\n{len(segment_results)} segment tests computed.")
    else:
        print("No segment columns found in data.")
else:
    print("No data for segment analysis.")

In [ ]:
# --- Charts ---

chart_paths = []

if control is not None and treatment is not None and primary_metric in control.columns:

    # Chart 1: Distribution comparison (histogram/KDE)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(control[primary_metric].dropna(), bins=50, alpha=0.5, color=BRAND_COLOURS["primary_blue"],
            label=f"Control (n={len(control)})")
    ax.hist(treatment[primary_metric].dropna(), bins=50, alpha=0.5, color=BRAND_COLOURS["green"],
            label=f"Treatment (n={len(treatment)})")
    ax.set_title(f"Distribution: {primary_metric}")
    ax.set_xlabel(primary_metric)
    ax.legend()
    plt.tight_layout()
    path = save_chart(fig, f"exp-distribution-{experiment_name}", report_date, output_dir)
    chart_paths.append(path)

    # Chart 2: Forest plot (effect sizes with CIs)
    all_tests = []
    if "primary" in results["tests"]:
        all_tests.append(results["tests"]["primary"])
    all_tests.extend(results["tests"].get("guardrails", []))

    if all_tests:
        fig, ax = plt.subplots(figsize=(10, max(4, len(all_tests) * 0.8)))
        y_pos = range(len(all_tests))
        names = [t["metric"] for t in all_tests]
        deltas = [t["absolute_delta"] for t in all_tests]
        ci_lows = [t["ci_lower"] for t in all_tests]
        ci_highs = [t["ci_upper"] for t in all_tests]
        colours = [BRAND_COLOURS["green"] if t["significant"] and t["absolute_delta"] > 0
                   else BRAND_COLOURS["red"] if t["significant"] and t["absolute_delta"] < 0
                   else BRAND_COLOURS["mid_grey"] for t in all_tests]

        for i, (name, delta, ci_lo, ci_hi, colour) in enumerate(zip(names, deltas, ci_lows, ci_highs, colours)):
            ax.plot([ci_lo, ci_hi], [i, i], color=colour, linewidth=2)
            ax.plot(delta, i, "o", color=colour, markersize=8)

        ax.axvline(x=0, color=BRAND_COLOURS["dark_navy"], linestyle="--", alpha=0.5)
        ax.set_yticks(list(y_pos))
        ax.set_yticklabels(names)
        ax.set_xlabel("Effect (treatment - control)")
        ax.set_title(f"Effect Sizes with 95% CI — {experiment_name}")
        plt.tight_layout()
        path = save_chart(fig, f"exp-forest-{experiment_name}", report_date, output_dir)
        chart_paths.append(path)

    # Chart 3: Segment breakdown bars
    if segment_results:
        # Group by segment column, show deltas
        seg_df = pd.DataFrame(segment_results)
        for seg_col_name in seg_df["segment_column"].unique():
            sub = seg_df[seg_df["segment_column"] == seg_col_name]
            fig, ax = plt.subplots(figsize=(10, 6))
            colours = [BRAND_COLOURS["green"] if d > 0 else BRAND_COLOURS["red"] for d in sub["absolute_delta"]]
            ax.barh(sub["segment_value"], sub["absolute_delta"], color=colours, alpha=0.8)
            ax.axvline(x=0, color=BRAND_COLOURS["dark_navy"], linestyle="--", alpha=0.5)
            ax.set_xlabel(f"Delta ({primary_metric})")
            ax.set_title(f"Treatment Effect by {seg_col_name} — {experiment_name}")
            plt.tight_layout()
            path = save_chart(fig, f"exp-segments-{seg_col_name}-{experiment_name}", report_date, output_dir)
            chart_paths.append(path)
            break  # Only first segment dimension as chart

    # Chart 4: Time series for novelty effects (if date column exists)
    date_col = None
    for candidate in ["date", "day", "created_at", "timestamp"]:
        if candidate in control.columns:
            date_col = candidate
            break
    if date_col:
        fig, ax = plt.subplots(figsize=(10, 6))
        for label, data, colour in [("Control", control, BRAND_COLOURS["primary_blue"]),
                                     ("Treatment", treatment, BRAND_COLOURS["green"])]:
            daily = data.groupby(date_col)[primary_metric].mean()
            ax.plot(range(len(daily)), daily.values, marker="o", linewidth=2, color=colour, label=label)
        ax.set_title(f"Daily {primary_metric} — Novelty Check")
        ax.set_xlabel("Day")
        ax.legend()
        plt.tight_layout()
        path = save_chart(fig, f"exp-novelty-{experiment_name}", report_date, output_dir)
        chart_paths.append(path)

    print(f"\n{len(chart_paths)} experiment charts generated.")
else:
    print("No data for charts.")

In [ ]:
# --- Export ---

if results.get("tests"):
    save_json(results, f"experiment-results-{experiment_name}", report_date, output_dir)

    # Generate summary markdown
    lines = [f"# {experiment_name} — Statistical Analysis\n"]
    lines.append(f"**Date:** {report_date}\n")

    if "primary" in results["tests"]:
        p = results["tests"]["primary"]
        sig_label = "Statistically significant" if p["significant"] else "Not statistically significant"
        lines.append(f"## Primary Metric: {p['metric']}\n")
        lines.append(f"| Measure | Value |")
        lines.append(f"|---------|-------|")
        lines.append(f"| Control mean | {p['control_mean']:.4f} (n={p['control_n']}) |")
        lines.append(f"| Treatment mean | {p['treatment_mean']:.4f} (n={p['treatment_n']}) |")
        lines.append(f"| Absolute delta | {p['absolute_delta']:.4f} |")
        lines.append(f"| Relative delta | {p.get('relative_delta_pct', 'N/A')}% |")
        lines.append(f"| p-value | {p['p_value']:.6f} |")
        lines.append(f"| 95% CI | [{p['ci_lower']:.4f}, {p['ci_upper']:.4f}] |")
        lines.append(f"| Significance | {sig_label} |")
        if "power" in p:
            lines.append(f"| Post-hoc power | {p['power']:.2%} |")
        if "mde_relative_pct" in p:
            lines.append(f"| MDE (80% power) | {p['mde_relative_pct']:.1f}% |")
        lines.append("")

    for gr in results["tests"].get("guardrails", []):
        status = "BREACHED" if gr["significant"] and gr["absolute_delta"] < 0 else "Safe"
        lines.append(f"## Guardrail: {gr['metric']} — {status}")
        lines.append(f"Delta: {gr['absolute_delta']:.4f}, p={gr['p_value']:.4f}\n")

    summary_md = "\n".join(lines)
    md_path = Path(output_dir or str(WORKSPACE_ROOT / "data" / "reports" / "charts")) / f"experiment-summary-{experiment_name}-{report_date}.md"
    md_path.parent.mkdir(parents=True, exist_ok=True)
    md_path.write_text(summary_md)
    print(f"Saved summary: {md_path}")
else:
    print("No results to export.")